In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.datasets import make_regression # used to generate a random sample for regression problems
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

---
## <u>Generate Dataset</u>

In [2]:
X, y = make_regression(   # no encoding or scaling need (scaling only needed if we make a penalization model like L1/L2)
    n_samples = 10000,    # no. of rows
    n_features = 15,      # no. of cols
    noise = 15,           # adding some noise in the data
    n_informative = 12,   # no. of informative cols
    random_state = 42
)

---
## <u>Train Test Split</u>

In [3]:
# the generated dataset is in the from of 1d np arrays so we convert into df
X = pd.DataFrame(X)

# Note : we do dont do (y = pd.DataFrame(y)) as the GradientBoostingRegressor wants a flat 1D array/series list

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

---
# <u>Create model, Train and Predict</u>

In [5]:
# making the pipeline to do hyperparamter tuning also

full_tree = DecisionTreeRegressor(random_state = 42)
full_tree.fit(X_train, y_train)

path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

reduced_ccp_alphas = ccp_alphas[::600] # shape = (14,)

steps = [("GBR", GradientBoostingRegressor(random_state = 42))]
pipeline = Pipeline(steps)

param_grid = {
    "GBR__n_estimators" : [100, 200, 300],         # No. of decision trees i.e. m value
    "GBR__learning_rate" : [0.01, 0.05, 0.1, 0.2],     
    "GBR__max_depth" : [2, 3, 4, 5],
    "GBR__subsample" : [0.7, 0.8, 0.9, 1.0],       # the fraction of samples to be used for fitting individual weak learners (DT)
    "GBR__ccp_alpha" : reduced_ccp_alphas   
}

GBR_cv = RandomizedSearchCV(
    pipeline,
    param_grid,
    cv = 5,
    n_iter = 20,
    n_jobs = -1,
    random_state = 42
)

GBR_cv.fit(X_train, y_train)
y_training_pred = GBR_cv.predict(X_train)
y_test_pred = GBR_cv.predict(X_test)

---
# <u>Evaluate</u>

In [8]:
print("For Gradient Boosting Regressor (hyperparameter tuning) :-\n")
print("Best Parameters : ", GBR_cv.best_params_)
print("\nTrain Accuracy (R2 Score) : ", r2_score(y_train, y_training_pred))
print("\nTest Accuracy (R2 Score) : ", r2_score(y_test, y_test_pred))

For Gradient Boosting Regressor (hyperparameter tuning) :-

Best Parameters :  {'GBR__subsample': 0.8, 'GBR__n_estimators': 200, 'GBR__max_depth': 2, 'GBR__learning_rate': 0.2, 'GBR__ccp_alpha': np.float64(0.17126130461220782)}

Train Accuracy (R2 Score) :  0.9820019279297602

Test Accuracy (R2 Score) :  0.9744872583855215
